# Data General Processing

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

from fair_ed.config import load_paths
from fair_ed.data.features import display_outliers_count, remove_outliers



In [ ]:
df_master = pd.read_csv(os.path.join(PATHS['output_dir'], 'master_dataset.csv'))

In [ ]:
df_master["gender"].value_counts()

# 1. General filter - Age, triage_acuity

In [ ]:
print('Before filtering for "age" >= 18 : master dataset size = ', len(df_master))
df_master = df_master[df_master['age'] >= 18]
print('After filtering for "age" >= 18 : master dataset size = ', len(df_master))

In [ ]:
print('Before filtering for non-null "triage_acuity" >= 18 : master dataset size = ', len(df_master))
df_master = df_master[df_master['triage_acuity'].notnull()]
print('After filtering for non-null "triage_acuity" >= 18 : master dataset size = ', len(df_master))

# 2. Outlier Detection

In [ ]:
vitals_valid_range = {
    'temperature': {'outlier_low': 14.2, 'valid_low': 26, 'valid_high': 45, 'outlier_high':47},
    'heartrate': {'outlier_low': 0, 'valid_low': 0, 'valid_high': 350, 'outlier_high':390},
    'resprate': {'outlier_low': 0, 'valid_low': 0, 'valid_high': 300, 'outlier_high':330},
    'o2sat': {'outlier_low': 0, 'valid_low': 0, 'valid_high': 100, 'outlier_high':150},
    'sbp': {'outlier_low': 0, 'valid_low': 0, 'valid_high': 375, 'outlier_high':375},
    'dbp': {'outlier_low': 0, 'valid_low': 0, 'valid_high': 375, 'outlier_high':375},
    'pain': {'outlier_low': 0, 'valid_low': 0, 'valid_high': 10, 'outlier_high':10},
    'acuity': {'outlier_low': 1, 'valid_low': 1, 'valid_high': 5, 'outlier_high':5},
}

In [ ]:
display_outliers_count(df_master, vitals_valid_range)

In [ ]:
df_master = remove_outliers(df_master, vitals_valid_range)

In [ ]:
pd.set_option('display.max_columns', None)
df_master.head()

# 3. Dataset Split (train:0.8, test: 0.2, use seed to fix)

In [ ]:
df_train=df_master.sample(frac=0.8,random_state=10) 
df_test=df_master.drop(df_train.index)

In [ ]:
print('Training dataset size = ', len(df_train))
print('Testing dataset size = ', len(df_test))

In [ ]:
df_train.head()

4. Missing Value imputation

In [ ]:
df_missing_stats = df_train.isnull().sum().to_frame().T
df_missing_stats.loc[1] = df_missing_stats.loc[0] / len(df_train)
df_missing_stats.index = ['no. of missing values', 'percentage of missing values']
df_missing_stats

In [ ]:
# Vitals that need imputation
vitals_cols = [col for col in df_master.columns if len(col.split('_')) > 1 and 
                                                   col.split('_')[1] in vitals_valid_range and
                                                   col.split('_')[1] != 'acuity']
vitals_cols

In [ ]:
imputer = SimpleImputer(strategy='median')
df_train[vitals_cols] = imputer.fit_transform(df_train[vitals_cols])
df_test[vitals_cols] = imputer.transform(df_test[vitals_cols])

In [ ]:
# Labs that need imputation

lab_columns = [
    'ALBUMIN', 'ALKALINE PHOSPHATASE', 'ALT (SGPT)', 'ANION GAP', 'AST (SGOT)', 
    'BASOPHIL % (AUTO DIFF)', 'BILIRUBIN, TOTAL', 'BLOOD UREA NITROGEN (BUN)', 
    'BLOOD: URINE (UA)', 'BNP, NT-PRO', 'Basophil Absolute (Combined)', 'CHLORIDE', 
    'CO2', 'CREATININE', 'Calcium (Combined)', 'EOSINOPHIL % (AUTO DIFF)', 
    'Eosinophil Absolute (Combined)', 'GLOBULIN', 'GLUCOSE', 'GLUCOSE: URINE (UA)', 
    'HCT & Hemoglobin (Combined)', 'HEMATOCRIT (HCT)', 'HIGH SENSITIVITY TROPONIN', 
    'IMMATURE GRANULOCYTE % (AUTODIFF) WAM', 'IMMATURE GRANULOCYTE, ABSOLUTE (AUTO DIFF) WAM', 
    'INFLUENZA A', 'INFLUENZA B', 'INR', 'KETONE: URINE (UA)', 'LEUKOCYTE ESTERASE: URINE (UA)', 
    'LIPASE', 'LYMPHOCYTE % (AUTO DIFF)', 'Lymphocyte Absolute (Combined)', 'MAGNESIUM', 
    'MEAN CORPUSCULAR VOLUME (MCV)', 'MONOCYTE % (AUTO DIFF)', 'Monocyte Absolute (Combined)', 
    'NEUTROPHIL % (AUTO DIFF)', 'NEUTROPHIL, ABSOLUTE (AUTO DIFF)', 'NITRITE: URINE (UA)', 
    'NRBC (Combined)', 'PART. THROMBOPLASTIN TIME', 'PCO2, Venous (Combined)', 
    'PH: URINE (UA)', 'PLATELET COUNT (PLT)', 'PO2, Venous (Combined)', 
    'POC: BHCG, ISTAT QUANTITATIVE', 'POC: HCG, ISTAT QUALITATIVE', 'POC:BASE EXCESS, ISTAT', 
    'POC:GLUCOSE BY METER', 'POC:HCO3 (V), ISTAT', 'POC:LACTATE, ISTAT', 
    'POC:O2 SATURATION, ISTAT VENOUS', 'POC:TCO2 (V), ISTAT', 'PROTEIN, TOTAL', 
    'PROTEIN: URINE (UA)', 'PROTHROMBIN TIME', 'Potassium (Combined)', 
    'RED BLOOD CELLS (RBC)', 'RED CELL DISTRIBUTION WIDTH (RDW)', 'RSV', 'SARS-COV-2 RNA', 
    'SPECIFIC GRAVITY: URINE (UA)', 'Sodium (Combined)', 'TROPONIN I', 
    'WHITE BLOOD CELLS (WBC)', 'eGFR (Combined)', 'pH, Venous (Combined)'
]

existing_lab_columns = [col for col in lab_columns if col in df_master.columns]
print(f"Found {len(existing_lab_columns)} out of {len(lab_columns)} lab columns in the dataset")

df_labs = df_master[existing_lab_columns]

In [ ]:
# Print all lab testing with their percentage of missing values
print("=== All Lab Tests with Missing Value Percentages ===")
print(f"{'Lab Test Name':<50} {'Missing %':<10} {'Missing Count':<15} {'Available Count'}")

lab_missing_summary = []

for col in existing_lab_columns:
    missing_count = df_train[col].isnull().sum()
    available_count = len(df_train) - missing_count
    missing_pct = missing_count / len(df_train) * 100
    
    lab_missing_summary.append({
        'Lab': col,
        'Missing_Pct': missing_pct,
        'Missing_Count': missing_count,
        'Available_Count': available_count
    })
    
    print(f"{col:<50} {missing_pct:>7.1f}%   {missing_count:>10}     {available_count:>10}")

lab_missing_summary.sort(key=lambda x: x['Missing_Pct'], reverse=True)

print("\n" + "=" * 95)
print("=== Labs Sorted by Missing Percentage (Highest to Lowest) ===")
print(f"{'Rank':<5} {'Lab Test Name':<50} {'Missing %':<10} {'Available Data'}")
print("-" * 95)

for i, lab in enumerate(lab_missing_summary, 1):
    print(f"{i:<5} {lab['Lab']:<50} {lab['Missing_Pct']:>7.1f}%   {lab['Available_Count']:>10} patients")

print(f"\n=== Summary Statistics ===")
print(f"Total lab columns analyzed: {len(existing_lab_columns)}")
print(f"Training dataset size: {len(df_train)}")

ranges = [
    (100, 100, "100% missing"),
    (95, 99.9, "95-99% missing"), 
    (90, 94.9, "90-95% missing"),
    (80, 89.9, "80-90% missing"),
    (50, 79.9, "50-80% missing"),
    (0, 49.9, "0-50% missing")
]

for min_pct, max_pct, label in ranges:
    count = len([lab for lab in lab_missing_summary if min_pct <= lab['Missing_Pct'] <= max_pct])
    if count > 0:
        print(f"{label}: {count} columns")

# 5. Save processed train/test splits

In [ ]:
os.makedirs(PATHS['output_dir'], exist_ok=True)
df_train.to_csv(PATHS['train_path'], index=False)
df_test.to_csv(PATHS['test_path'], index=False)
